In [ ]:
#| default_exp search

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import ast, math, re, sys
from functools import lru_cache
import numpy as np
from fastcore.all import AttrDict, L, Path, chunked, defaults, first, ifnone, merge, patch, store_attr
from fastcore.parallel import ProcessPoolExecutor
from fastlite import Database
from apswutils.db import Table
from multiprocessing import get_context
from litesearch.core import (sql_in, rowid_sel, content_id, NP_DTYPE, process_content, write_txn, db_lock,
                             upsert_all, rrf_all)
from litesearch.topics import get_graph
from litesearch.utils import hash_embed
from vruksha.build import build_graph, resolve_entities, cooccur_edges

## Traversal, PPR and fusion

The graph becomes a third RRF leg beside FTS and vectors.

In [ ]:
#| export
def _adjacency(g, nodes, hops=2, max_nodes=4000):
    '''BFS the edge table out to `hops` from `nodes`; returns an undirected dict-of-dict adjacency.

    Read as raw tuples rather than through the table wrapper. A two-hop walk off twelve seeds pulls
    ~16k edges, and fastlite turns each one into a dict with a description lookup per row — 326k
    dicts over twenty queries, of which this needs three columns and none of the keys.'''
    adj, seen, frontier = {}, set(nodes), set(nodes)
    con, tbl = g.edges.db.conn, g.edges.name
    for _ in range(max(hops, 0)):
        if not frontier or len(seen) >= max_nodes: break
        fl = list(frontier)
        rows = []
        for b in chunked(fl, 400):
            rows += con.execute(f"select src, dst, weight from {tbl} "
                                f"where {sql_in('src', b)} OR {sql_in('dst', b)}").fetchall()
        nxt = set()
        for s, d, w in rows:
            w = w or 1.0
            adj.setdefault(s, {})[d] = max(adj.setdefault(s, {}).get(d, 0.0), w)
            adj.setdefault(d, {})[s] = max(adj.setdefault(d, {}).get(s, 0.0), w)
            for x in (s, d):
                if x not in seen: nxt.add(x)
        seen |= nxt
        frontier = nxt
    return adj

def _csr(adj, order):
    '''`(src, dst, w)` index arrays for `adj`, row-normalised — the transition matrix as three arrays.

    Built once and reused by every power iteration, which is the whole point: the walk itself is
    then twelve numpy passes instead of twelve nested python loops over the same unchanging edges.'''
    src, dst, wt = [], [], []
    for u, nb in adj.items():
        iu, s = order[u], (sum(nb.values()) or 1.0)
        for v, w in nb.items(): src.append(iu); dst.append(order[v]); wt.append(w/s)
    return (np.array(src, dtype=np.intp), np.array(dst, dtype=np.intp),
            np.array(wt, dtype=np.float64))

def _ppr(adj, seeds, damping=0.85, iters=12):
    '''Personalized PageRank over a dict-of-dict adjacency.

    The iteration is `r <- d * Wᵀr + (1-d) * p0` and nothing but `r` changes between rounds, so the
    edges are flattened into index arrays once and each round becomes a gather plus a `bincount` —
    the same sum, done by numpy instead of by a dict lookup per edge per round. Dangling nodes still
    drop their mass rather than redistributing it, exactly as the dict version did.'''
    if not seeds: return {}
    order = {}
    for u, nb in adj.items():
        if u not in order: order[u] = len(order)
        for v in nb:
            if v not in order: order[v] = len(order)
    for k in seeds:
        if k not in order: order[k] = len(order)
    n = len(order)
    p0 = np.zeros(n)
    tot = sum(seeds.values()) or 1.0
    for k, v in seeds.items(): p0[order[k]] = v/tot
    src, dst, wt = _csr(adj, order)
    wt = wt * damping
    r, rest = p0, (1-damping)*p0
    for _ in range(iters):
        r = np.bincount(dst, weights=r[src]*wt, minlength=n) + rest
    return {k: float(r[i]) for k, i in order.items() if r[i]}

In [ ]:
#| export
def _canon_mentions(g,                # graph tables from `get_graph`
                    col,              # 'chunk_id' or 'entity_id' — the side being filtered
                    vals,             # values to filter on
                    use_canon=True,   # resolve entity_id to its canonical id
                    batch=400):
    '''`(chunk_id, entity_id)` for the matching mentions, with `entity_id` already canonicalised.'''
    con, mn, en = g.mentions.db.conn, g.mentions.name, g.entities.name
    sel = (f'select m.chunk_id, coalesce(nullif(e.canon, \'\'), m.entity_id) from {mn} m '
           f'left join {en} e on e.id = m.entity_id') if use_canon else \
          f'select m.chunk_id, m.entity_id from {mn} m'
    out = []
    for b in chunked(vals, batch):
        out += con.execute(f"{sel} where {sql_in('m.'+col, b)}").fetchall()
    return out

### `graph_search`

Seeds the walk with the top hybrid hits, spreads PageRank mass over the entity graph, and fuses
what it reaches as a third leg. The walk returns nothing at any dead end, and RRF ignores an empty
list, so the fallback is plain hybrid search with no branch for it.

In [ ]:
#| export
def _leg(db, g, seeds_from, cols, table_name, limit, hops, damping, iters, use_canon):
    'Chunks the walk reaches, ranked by PPR mass. Empty at every dead end, which RRF then ignores.'
    if not seeds_from: return []
    # canon is resolved by joining the mentions being read rather than by loading the whole entity
    # table into a dict. Every query paid O(entities) for a map it used O(mentions-of-12-chunks) of,
    # which is the one cost here that grew with the corpus rather than with the query.
    seeds = {}
    for _, e in _canon_mentions(g, 'chunk_id', seeds_from, use_canon): seeds[e] = seeds.get(e, 0.0) + 1.0
    if not seeds: return []
    mass = _ppr(_adjacency(g, set(seeds), hops), seeds, damping, iters)
    eids = [e for e, m in sorted(mass.items(), key=lambda kv: -kv[1])[:200] if m > 0]
    if not eids: return []
    inv = {}
    for cid, e in _canon_mentions(g, 'entity_id', eids, use_canon):
        inv[cid] = inv.get(cid, 0.0) + mass.get(e, 0.0)
    if not inv: return []
    ranked = sorted(inv.items(), key=lambda kv: -kv[1])[:limit*3]
    sel = ','.join([rowid_sel() if c == 'rowid' else c for c in cols])
    rows = {r['id']: r for r in db.t[table_name](select=sel, where=sql_in('id', [c for c, _ in ranked]))}
    return [rows[c] for c, _ in ranked if c in rows]

@patch
def graph_search(self:Database,
                 q:str,                 # query string
                 emb:bytes,             # query embedding
                 columns:list=None,     # columns to return
                 limit:int=20,          # max results
                 table_name='store',    # chunk store
                 prefix=None,           # graph table prefix
                 seed_n:int=12,         # hybrid hits used to seed the graph walk
                 hops:int=2,            # edge-table BFS depth
                 damping:float=0.85,    # PPR damping
                 iters:int=10,          # PPR iterations
                 graph_w:float=0.5,     # weight of the graph leg, low by default. See the docstring.
                 rrf_k:int=60,
                 use_canon=True,
                 **kw):                 # forwarded to Database.search
    'Hybrid search plus a graph leg: PPR over the entity graph seeded by the top hybrid hits.'
    g = self.get_graph(table_name, prefix)
    cols = list(columns or [])
    if 'rowid' not in cols: cols = ['rowid'] + cols
    if 'id' not in cols: cols = cols + ['id']
    base = self.search(q, emb, columns=cols, limit=max(seed_n*3, limit), table_name=table_name,
                       rrf=False, **kw)
    if not base: return []
    fts, vec = base['fts'], base['vec']
    seeds = [r['id'] for r in rrf_all([fts, vec], rrf_k, seed_n) if r.get('id')]
    leg = _leg(self, g, seeds, cols, table_name, limit, hops, damping, iters, use_canon)
    return rrf_all([fts, vec, leg], rrf_k, limit, weights=[1.0, 1.0, graph_w])


In [ ]:
#| export
def graph_stats(db, store='store', prefix=None):
    'Row counts and top-degree nodes for a built graph.'
    g = db.get_graph(store, prefix)
    ne = first(db.q(f'select count(*) c from {g.entities.name}'))['c']
    nm = first(db.q(f'select count(*) c from {g.mentions.name}'))['c']
    ng = first(db.q(f'select count(*) c from {g.edges.name}'))['c']
    nc = first(db.q(f'select count(distinct canon) c from {g.entities.name}'))['c']
    top = db.q(f'''select e.content, e.kind, count(*) d from {g.edges.name} g
                   join {g.entities.name} e on e.id=g.src group by g.src order by d desc limit 10''')
    return dict(entities=ne, canonical=nc, mentions=nm, edges=ng, top_degree=top)

## End to end: code

In [ ]:
import os, tempfile
from litesearch import database
from litesearch.data import dir2chunks
from litesearch.topics import topic_nodes
from litesearch.utils import hash_embed
from fastcore.all import Path

_chunks = dir2chunks('../vruksha', file_glob='*.py')

_tmp = tempfile.mkdtemp()
emb = lambda txts, **kw: hash_embed(txts)
db = database(f'{_tmp}/code.db')
store = db.get_store(hash=True, ann=True)
rows = [dict(content=c['content'], metadata=str(c['metadata'])) for c in _chunks if c['content'].strip()]
store.insert_all([dict(r, embedding=e.tobytes()) for r,e in zip(rows, emb([r['content'] for r in rows]))],
                 upsert=True, hash_id='id', hash_id_columns=['content'])
store.rebuild_index()

print('build   :', build_graph(db, _chunks, prose=False, emb_fn=emb))
print('resolve :', resolve_entities(db))
print('topics  :', topic_nodes(db))
st = graph_stats(db)
print('stats   :', {k:v for k,v in st.items() if k!='top_degree'})
for t in st['top_degree'][:6]: print(f"   {t['d']:3d}  {t['content'][:44]}")
assert st['entities'] > 100 and st['edges'] > 50

build   : {'entities': 106, 'mentions': 156, 'edges': 148, 'windows': 29}
resolve : {'merged': 0, 'by_ann': 0, 'by_lexical': 0, 'edges': 148, 'entities': 106, 'resolvable': 0, 'canonical': 106}
topics  : {'topics': 4, 'method': 'knn'}
stats   : {'entities': 110, 'canonical': 110, 'mentions': 170, 'edges': 148}
    23  build_graph
     8  _ann_pairs
     8  _toks
     7  resolve_entities
     7  _windowstore
     6  _adjacency


The hub symbols are real ones, because the edges are AST calls rather than guesses.

## End to end: PDF

In [ ]:
from litesearch.data import file_parse, repo_root
from vruksha.build import _pmi_edges
# `nbdev_docs`/`nbdev_preview` execute notebooks from the *project root*, not from `nbs/`,
# so a bare relative path resolves under test and not under the docs build.
pchunks = [c for c in file_parse(repo_root()/'nbs/pdfs/attention_is_all_you_need.pdf') if c['content'].strip()]
print('chunks:', len(pchunks))

pdb = database(f'{_tmp}/pdf.db')
pstore = pdb.get_store(hash=True, ann=True)
prows = [dict(content=c['content'], metadata=str(c['metadata'])) for c in pchunks]
pstore.insert_all([dict(r, embedding=e.tobytes()) for r,e in zip(prows, emb([r['content'] for r in prows]))],
                  upsert=True, hash_id='id', hash_id_columns=['content'])
pstore.rebuild_index()

print('build   :', build_graph(pdb, pchunks, code=False, emb_fn=emb))
print('resolve :', resolve_entities(pdb))
pst = graph_stats(pdb)
print('stats   :', {k:v for k,v in pst.items() if k!='top_degree'})
print('top entities:', [t['content'][:28] for t in pst['top_degree'][:8]])

# contrast: the same PMI over page-sized windows
canon = {r['id']:(r['canon'] or r['id']) for r in pdb.t.entities(select='id, canon')}
per_chunk = {}
for m in pdb.t.mentions(select='chunk_id, entity_id'):
    per_chunk.setdefault(m['chunk_id'], set()).add(canon.get(m['entity_id'], m['entity_id']))
print(f"edges, sentence windows : {pst['edges']}")
print(f"edges, page windows     : {len(_pmi_edges(list(per_chunk.values())))}   <- clique blowup")
assert pst['entities'] > 50

Dictionary used where Stream expected, treating as empty stream


Dictionary used where Stream expected, treating as empty stream


Dictionary used where Stream expected, treating as empty stream


Dictionary used where Stream expected, treating as empty stream


chunks: 98


build   : {'entities': 913, 'mentions': 1125, 'edges': 160, 'windows': 245}
resolve : {'merged': 274, 'by_ann': 138, 'by_lexical': 136, 'edges': 103, 'entities': 913, 'resolvable': 913, 'canonical': 639}
stats   : {'entities': 913, 'canonical': 639, 'mentions': 1125, 'edges': 103}
top entities: ['attention', 'models', 'tokens', 'sentence', 'positions', 'warmup steps', 'pos', 'processing systems']
edges, sentence windows : 103
edges, page windows     : 110   <- clique blowup


In [ ]:
#| hide
# a graph built over a doc/tree store must reference the store's real (composite) chunk ids, so
# graph_search can join mentions back to chunks. Feed store rows *with* their id.
from vruksha import build_graph
_gdb = database()
_gdb.add_doc([(0, "# Rights\n\nThe supplier shall inform the consumer of the right of withdrawal.\n\n"
                 "# Duties\n\nThe consumer may exercise the right of withdrawal within the stated period.")],
             title='GraphDoc', emb_fn=lambda ts, **kw: hash_embed(ts, 256))
_grows = list(_gdb.t['store'](select='id, content'))
build_graph(_gdb, _grows, emb_fn=lambda ts, **kw: hash_embed(ts, 256))
_ids = {r['id'] for r in _grows}
_cids = {m['chunk_id'] for m in _gdb.get_graph('store').mentions(select='chunk_id')}
assert _cids and _cids <= _ids, 'graph mentions must reference real store chunk ids'


In [ ]:
#| hide
# `graph=` is litesearch's seam and vruksha's implementation, so the assertions live here.
from litesearch.core import database
from litesearch.utils import hash_embed
from vruksha import build_graph
import numpy as np

_cdb = database()
_cenc = lambda ts, **kw: hash_embed(ts, 256)
_cdb.add_doc([(0, "# Rights\n\n## Article 1\n\nThe consumer has a right of withdrawal within fourteen days, "
                 "subject to Article 2 and the definitions therein.\n\n## Article 2\n\nDefinitions: a consumer "
                 "means a natural person acting outside their trade.\n\n# Duties\n\n## Article 3\n\nThe supplier "
                 "shall inform the consumer of the right of withdrawal before the contract is concluded.")],
             title='ConsumerRules', emb_fn=_cenc)
_cemb = hash_embed(['right of withdrawal'], 256)[0].astype(np.float16).tobytes()
_ctx = _cdb.context('right of withdrawal', _cemb, sections=3, related=5)
assert _ctx.results, 'context returned no operative sections'
assert all('›' in r.breadcrumb for r in _ctx.results), 'each result is a provision with a breadcrumb path'
assert _ctx.results[0].text and _ctx.results[0].tree is not None, 'a result carries full text and tree context'
assert isinstance(_ctx.related, L)
build_graph(_cdb, list(_cdb.t['store'](select='id, content')), emb_fn=_cenc)
# graph= is opt-in, so the graph leg needs asking for by name; without this the default
# path stopped covering it at all
assert _cdb.context('right of withdrawal', _cemb, sections=3, related=5, graph=True).results
assert _cdb.context('right of withdrawal', _cemb, sections=3, related=5).results
assert not any(r.via == 'graph' for r in _cdb.context('right of withdrawal', _cemb, related=5).related)

assert _cdb.context('right of withdrawal', _cemb, related=5, graph=True, ann=True).results


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()